# Exercise 2.1: Loading Data & First Diagnostics (Angola IEA)

This notebook uses `IEA_2025_IV_TRIM_IND.sav`, the individual file of the
Inquerito ao Emprego em Angola (IEA), 4th quarter 2025, published by INE Angola.

It is a real SPSS export: 53,353 people, 206 columns, all variable and value
labels in Portuguese.

You will practice:
- Loading an SPSS file with `pd.read_spss()`
- Deciding whether to apply the file's value labels, and seeing what that costs
- Reading the codebook that ships inside the file
- Loading only the columns you need with `usecols`
- Running structural diagnostics with `info()`, `describe()` and `value_counts()`
- Spotting sentinel codes and empty columns before any cleaning

> **Pipeline:** this notebook only reads `0_raw/`. It writes nothing.

### Path Setup (run first)

> Use `os.path.join` for path construction.
> Required base path: `DATA_RAW_DIR = '../../data/0_raw/angola/employment_survey'`.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola/employment_survey'
RAW_FILE = 'IEA_2025_IV_TRIM_IND.sav'
raw_path = os.path.join(DATA_RAW_DIR, RAW_FILE)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Data path:', raw_path)
print('Exists?:', os.path.exists(raw_path))

---

## Task 1: Load the file and take a first look

`pd.read_spss()` reads SPSS `.sav` files. By default it applies the value labels
stored in the file, so coded variables come back as readable text.

In [ ]:
df_labelled = pd.read_spss(raw_path)

print('Shape:', df_labelled.shape)
df_labelled[['PROV', 'AREA_RESID', 'DEM_SEX', 'DEM_AGE']].head()

In [ ]:
df_labelled[['PROV', 'AREA_RESID', 'DEM_SEX']].tail()

In [ ]:
df_labelled[['PROV', 'AREA_RESID', 'DEM_SEX', 'DEM_AGE']].sample(5, random_state=0)

**Answers:**

- The file holds 53,353 rows and 206 columns, one row per person interviewed.
- `PROV`, `AREA_RESID` and `DEM_SEX` arrive as readable Portuguese text
  (`Luanda`, `Urbana`, `Feminino`) rather than the numeric codes actually stored
  in the file, because `pd.read_spss` applies value labels by default.
- 206 columns is far more than any single analysis needs, which is what Task 4
  addresses.

---

## Task 2: The cost of applying value labels

Value labels are convenient, but they are applied to **every** labelled variable,
including ones where the label is a sentinel rather than a category. Reload with
`convert_categoricals=False` and compare the dtypes.

In [ ]:
df_raw = pd.read_spss(raw_path, convert_categoricals=False)

comparison = pd.DataFrame({
    'labelled': df_labelled[['PROV', 'DEM_SEX', 'MJT_SYR', 'DEM_AGE']].dtypes,
    'raw_codes': df_raw[['PROV', 'DEM_SEX', 'MJT_SYR', 'DEM_AGE']].dtypes,
})
comparison

In [ ]:
# MJT_SYR is the year the person started their main job.
print('Labelled, first 5 values:')
print(df_labelled['MJT_SYR'].head().tolist())
print()
print('Raw codes, describe:')
print(df_raw['MJT_SYR'].describe())

**Answers:**

- `MJT_SYR` is a **year**, but the labelled load returns it as `category`. That
  happens because the value `9997` carries the label `NAO SABE`, so pandas treats
  the whole column as categorical. A year you cannot subtract is useless.
- The raw load returns `float64` and `describe()` immediately exposes the damage:
  a mean of 3164.6 against a real range of 1965 to 2025.
- The rest of this series uses `convert_categoricals=False`, so the codes stay
  numeric and we decode them ourselves in 2.4. That is the same choice the Stata
  callout in the lesson recommends with `convert_categoricals=False`.

---

## Task 3: Read the codebook that ships inside the file

An SPSS file carries its own documentation: a label for every variable, and a
label for every coded value. The lesson shows this for Stata with
`variable_labels()`. For SPSS the equivalent lives in `pyreadstat`, and it can be
read **without loading the data**.

> This is the only cell in the whole series that imports `pyreadstat` directly.

In [ ]:
import pyreadstat

_, meta = pyreadstat.read_sav(raw_path, metadataonly=True)

for name in ['PROV', 'AREA_RESID', 'DEM_SEX', 'DEM_AGE', 'WKT_USHRSTOT', 'MJT_SYR']:
    print(f'{name:15s} {meta.column_names_to_labels[name]}')

In [ ]:
# Value labels: the code to text mapping behind each categorical variable
label_set = meta.variable_to_label['PROV']
print('PROV has', len(meta.value_labels[label_set]), 'provinces')
print(meta.value_labels[label_set])

**Answers:**

- Without the codebook, `WKT_USHRSTOT` is meaningless. With it, we know it is
  "Quantas horas o(a) nome geralmente trabalha por semana no total?", the usual
  weekly hours across all jobs.
- `PROV` maps 21 codes, 10 to 30, onto province names. Note that Angola created
  three new provinces in 2024: Icolo e Bengo (28), Moxico Leste (29) and Cuando
  (30). Any lookup table built before 2024 will be missing them, which is exactly
  the merge problem waiting in 2.5.
- Reading metadata only is fast because no data is loaded, so it is the cheapest
  possible first step with an unfamiliar file.

---

## Task 4: Load only the columns you need with `usecols`

206 columns is more than this analysis needs. `usecols` tells the reader to skip
the rest entirely, so they never enter memory.

The 29 columns below carry the survey's demographic core, its labour module, the
interview date, and the survey weight.

In [ ]:
SPSS_COLS = [
    'NIDF', 'PPNO', 'G_06_ID_IEA', 'PROV', 'AREA_RESID', 'G_15_TRIMESTRE',
    'DEM_REL', 'DEM_SEX', 'DEM_AGE', 'DEM_MRT', 'DEM_EDL', 'S03_01',
    'ATW_PAY', 'ATW_PFT', 'ATW_FAM', 'ABS_JOB',
    'SRH_JOB', 'SRH_BUS', 'SRH_AVN', 'SRH_AVL', 'SRH_DES',
    'WKT_USHRSTOT', 'WKT_ACHRSTOT', 'MJT_SYR', 'MJJ_EMP_REL', 'GHVEDT',
    'POND_IEA_IV_TRIM_2025_IND', 'G_12', 'G_13',
]

df = pd.read_spss(raw_path, usecols=SPSS_COLS, convert_categoricals=False)
print('Full file: ', df_raw.shape)
print('Subset:    ', df.shape)
df.head()

In [ ]:
full_mb = df_raw.memory_usage(deep=True).sum() / 1e6
subset_mb = df.memory_usage(deep=True).sum() / 1e6
print(f'Memory full:   {full_mb:8.2f} MB')
print(f'Memory subset: {subset_mb:8.2f} MB')
print(f'Saved:         {(1 - subset_mb / full_mb) * 100:8.1f}%')

**Answers:**

- The subset keeps all 53,353 rows but only 29 of 206 columns, cutting memory by
  roughly 90%.
- `usecols` skips columns at read time. Loading everything and then selecting
  with `df[COLS]` produces the same table but pays the full memory and time cost
  first, which matters when the file is larger than this one.
- A misspelled name in `SPSS_COLS` raises an error rather than being ignored,
  which is what you want.

---

## Task 5: Structural health check with `info()`

`df.info()` is the first diagnostic. Read the Non-Null Count column carefully:
it is where empty columns and skip patterns show up.

In [ ]:
df.info()

In [ ]:
missing = pd.DataFrame({
    'n_missing': df.isna().sum(),
    'pct_missing': (df.isna().mean() * 100).round(1),
}).sort_values('pct_missing', ascending=False)
missing

**Answers:**

- `G_12` and `G_13` are **100% missing**: 53,353 nulls out of 53,353 rows. They
  should hold household size and the number of adults, so they look useful in a
  column list and are worth nothing in practice. They get dropped in 2.3.
- The labour module columns (`MJT_SYR`, `WKT_USHRSTOT`, `MJJ_EMP_REL`) are around
  78% missing, and `SRH_AVL` is 98.5% missing. This is **not** damage. Those
  questions are only asked of people the questionnaire routes to them. Missing by
  design and missing by error need completely different treatment, which is the
  main lesson of 2.3.
- Everything is `float64`, including the identifiers `NIDF` and `PPNO` and the
  date `GHVEDT`. All three are wrong types, fixed in 2.2.

---

## Task 6: Summary statistics with `describe()`

`describe()` exposes impossible values. Look hard at every `min` and `max`.

In [ ]:
df[['DEM_AGE', 'WKT_USHRSTOT', 'WKT_ACHRSTOT', 'MJT_SYR', 'GHVEDT']].describe().T

In [ ]:
df.describe(include='all').T

**Answers:**

- `MJT_SYR` has a mean of 3164.6 and a max of 9997. `9997` is the "NAO SABE"
  sentinel and there are 1,673 of them, dragging the mean nearly 1,150 years into
  the future.
- `WKT_USHRSTOT` maxes at 997, the same sentinel pattern. Its real maximum is 120
  hours a week, which is itself implausible and gets a validation rule in 2.3.
- `DEM_AGE` runs from 0 to 120. Age 0 is legitimate: 1,532 infants. Age 120 is
  not.
- `GHVEDT` has a mean around 20,251,000, which is nonsense as a number because it
  is really the date 2025-12-04 stored as the digits `20251204`.

---

## Task 7: Explore categories with `value_counts()`

`value_counts()` is the fastest way to see what is actually in a coded column.
Always pass `dropna=False` so the gaps are counted too.

In [ ]:
print(df['PROV'].value_counts(dropna=False).sort_index())

In [ ]:
print(df['AREA_RESID'].value_counts(dropna=False))
print()
print(df['DEM_SEX'].value_counts(dropna=False))
print()
print(df['DEM_EDL'].value_counts(dropna=False).sort_index())

In [ ]:
print('Distinct households:', df['NIDF'].nunique())
print('Rows per household, describe:')
print(df['NIDF'].value_counts().describe())

**Answers:**

- All 21 province codes appear, from 10 to 30, with Luanda (14) the largest at
  4,424 people.
- `AREA_RESID` splits 34,173 urban and 19,180 rural. `DEM_SEX` splits 25,601 male
  and 27,752 female.
- `DEM_EDL` is 56.6% missing and its codes jump from 7 to 9, with no 8. Reading
  the codebook explains it: 9 means "Nenhum nivel", no level, so it is not an
  ordinal step above 7.
- There are 13,036 households across 53,353 people, a mean of 4.09 people each.
  Remember this number: 2.4 derives household size a different way and gets 5.58,
  for a reason worth understanding.

---

## Task 8: A quick visual sweep

Histograms of every numeric column at once are a fast way to spot sentinels: they
appear as a lonely spike far to the right of everything else.

In [ ]:
df[['DEM_AGE', 'WKT_USHRSTOT', 'WKT_ACHRSTOT', 'MJT_SYR',
    'DEM_EDL', 'POND_IEA_IV_TRIM_2025_IND']].hist(bins=30, figsize=(14, 8))
plt.tight_layout()
plt.show()

**Answers:**

- `MJT_SYR` and `WKT_USHRSTOT` both show a tiny bar at the far right, isolated
  from the rest of the distribution. That shape is the visual signature of a
  sentinel code.
- `DEM_AGE` is heavily skewed towards the young, which is expected for Angola:
  the median age in this sample is 17.
- None of these columns is ready for analysis yet. 2.2 fixes the types and 2.3
  removes the sentinels.